# Bộ dữ liệu đa hành vi

Notebook đọc bộ dữ liệu trong `data/multibehavior/`, không ghi đè dữ liệu. Chạy pipeline và lệnh xác minh theo [README](../README.md) trước; xem [giao thức dữ liệu](../docs/DATA_PROTOCOL.md) để hiểu nhãn mua 7 ngày và lịch sử tại thời điểm query. Ví dụ bên dưới dùng validation; test dành cho đánh giá mô hình cuối.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.dataset import require_complete, history_for_query
DATA = require_complete(ROOT / "data/multibehavior")
summary = json.loads((ROOT / "reports/multibehavior/summary.json").read_text(encoding="utf-8"))
pd.DataFrame(summary["split_summary"]).T

## Đọc riêng query, target và feature

Một query có thể có nhiều target hoặc không có target. Không đưa `target_count`, `label_end`, `has_purchase_target` hoặc `eligible_target_count` vào feature mô hình. Bảng target chứa nhãn, không được dùng làm danh sách ứng viên đầu vào cho bước chấm điểm.

In [ ]:
queries = pd.read_parquet(DATA / "splits/validation_queries.parquet")
targets = pd.read_parquet(DATA / "splits/validation_targets.parquet")
features = pd.read_parquet(DATA / "query_features.parquet")
print("Queries:", len(queries), "Target pairs:", len(targets))
print("Feature columns:", features.columns.tolist())

## Ví dụ lấy lịch sử đã quan sát

History dùng sự kiện của cùng visitor có `event_time <= query_time`, loại session `boundary_drop`. Giao dịch tại query_time đã được quan sát; chỉ giao dịch sau query_time, trong cửa sổ 7 ngày và cùng split mới có thể là target.

In [ ]:
events = pd.read_parquet(DATA / "events.parquet")
query = queries.iloc[0]
history = history_for_query(events, query)
assert history.event_time.le(query.query_time).all()
print("History events:", len(history))
history[["event_time", "event", "itemid", "item_idx"]].tail(10)

## Đọc kết quả kiểm tra đã lưu

Cell dưới đọc kết quả kiểm tra của lần tạo dữ liệu trong summary.json, không chạy lại toàn bộ kiểm tra trên Parquet. Độ phủ candidate không phải Recall/NDCG của mô hình. Query không có nhãn mua vẫn được giữ; ranking metric trên query có nhãn mua cần báo cáo mẫu số riêng.

In [ ]:
assert all(summary["checks"].values())
pd.Series(summary["checks"], name="passed")

## Cách dùng các bảng cho mô hình

Train interactions phù hợp cho mô hình fit tại cuối train; behavior edges hoặc history có thể dùng cho mô hình đa hành vi. Với ví dụ huấn luyện tại thời điểm query, chỉ dùng lịch sử đã quan sát tại thời điểm đó. Các mô hình được so sánh trực tiếp phải dùng cùng query, target và chính sách candidate. Hàm `candidates_at_time()` trong `scripts/dataset.py` dựng candidate tham chiếu, độc lập với target. Các mô hình nêu trong README hiện là hướng thực nghiệm, chưa có kết quả trong repo.